### **DLT - Databricks Declarative Pipelines**

In [0]:
# Expectations

my_rule = {
    "rule1": "product_id IS NOT NULL",
    "rule2": "product_name IS NOT NULL"
}

In [0]:
# Streaming Table 

import dlt
from pyspark.sql.functions import *

@dlt.table(name = "DimProducts_stage")

@dlt.expect_all_or_drop(my_rule)
def DimProducts_stage():
    df = spark.readStream.option("skipChangeCommits", "true").table("databricksete_cat.silver.products_silver") # Ignores overwrite commits
                
    return df
  

In [0]:
#Streaming view 
@dlt.view

def DimProducts_view():
    df = spark.readStream.table("LIVE.DimProducts_stage")
    return df

### **DimProducts**

In [0]:
# Created streaming table
dlt.create_streaming_table("DimProducts")

In [0]:
dlt.apply_changes(
    target = "DimProducts",
    source = "DimProducts_view",
    keys = ["product_id"],
    sequence_by = "product_id",
    stored_as_scd_type = 2
)